In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.utils import degree, homophily
from sklearn.model_selection import StratifiedShuffleSplit
import warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [3]:
#  File Paths 
DATA_PROCESSED = Path("../data/processed")

paysim_path = DATA_PROCESSED / "paysim_graph_v2.pt"
elliptic_path = DATA_PROCESSED / "elliptic_graph_v2.pt"
synthetic_path = DATA_PROCESSED / "synthetic_graph_v2.pt"

print("Environment setup complete.")

Environment setup complete.


## 1. Load Standardized Datasets

In [4]:
def load_graph(path, name):
    if path.exists():
        data = torch.load(path, weights_only=False)
        print(f" Loaded {name}: {data.num_nodes:,} nodes, {data.edge_index.shape[1]:,} edges.")
        return data
    else:
        print(f" {name} file not found at {path}. Skipping.")
        return None

paysim_data = load_graph(paysim_path, "PaySim")
elliptic_data = load_graph(elliptic_path, "Elliptic")
synthetic_data = load_graph(synthetic_path, "Synthetic")

 Loaded PaySim: 592,288 nodes, 325,933 edges.
 Loaded Elliptic: 203,769 nodes, 234,355 edges.
 Loaded Synthetic: 5,000 nodes, 40,900 edges.


## 2. Structural Analysis

In [6]:
def check_connectivity(data, name):
    num_nodes = data.num_nodes
    
    # Calculate degree (in + out)
    d = degree(data.edge_index[0], num_nodes=num_nodes) + \
        degree(data.edge_index[1], num_nodes=num_nodes)
    
    isolated = (d == 0).sum().item()
    pct_isolated = (isolated / num_nodes) * 100
    
    print(f"{name} Connectivity ")
    print(f"  Avg Degree: {d.mean():.2f}")
    print(f"  Max Degree: {d.max().item()}")
    print(f"  Isolated Nodes: {isolated:,} ({pct_isolated:.2f}%)")
    
    if pct_isolated > 20:
        print("   WARNING: High percentage of isolated nodes. Message passing may be limited.")

if paysim_data: check_connectivity(paysim_data, "PaySim")
if elliptic_data: check_connectivity(elliptic_data, "Elliptic")
if synthetic_data: check_connectivity(synthetic_data, "Synthetic")

PaySim Connectivity 
  Avg Degree: 1.10
  Max Degree: 9.0
  Isolated Nodes: 0 (0.00%)
Elliptic Connectivity 
  Avg Degree: 2.30
  Max Degree: 473.0
  Isolated Nodes: 0 (0.00%)
Synthetic Connectivity 
  Avg Degree: 16.36
  Max Degree: 83.0
  Isolated Nodes: 0 (0.00%)


In [7]:
def calculate_homophily(data, name):
    """
    Calculates edge homophily ratio: fraction of edges connecting nodes of same class.
    Excludes 'unknown' classes (-1) if present.
    """
    mask_src = data.y[data.edge_index[0]] >= 0
    mask_dst = data.y[data.edge_index[1]] >= 0
    valid_edge_mask = mask_src & mask_dst
    
    if valid_edge_mask.sum() == 0:
        print(f"  {name}: No labeled edges found for homophily calc.")
        return

    # Filtered edge index and y
    subset_edge_index = data.edge_index[:, valid_edge_mask]
    
    # Calculate homophily manually for clarity
    # y[src] == y[dst]
    src_y = data.y[subset_edge_index[0]]
    dst_y = data.y[subset_edge_index[1]]
    
    matches = (src_y == dst_y).sum().item()
    total = subset_edge_index.shape[1]
    
    ratio = matches / total
    
    print(f"\n {name} Homophily ")
    print(f"  Edge Homophily: {ratio:.4f}")
    if ratio > 0.8:
        print("  High clustering (Nodes connect to same class). GNNs usually excel here.")
    elif ratio < 0.2:
        print("  Heterophily (Nodes connect to opposite class). GNNs might need adjustments (e.g., GAT).")
    else:
        print("  Mixed connectivity.")

if paysim_data: calculate_homophily(paysim_data, "PaySim")
if elliptic_data: calculate_homophily(elliptic_data, "Elliptic")
if synthetic_data: calculate_homophily(synthetic_data, "Synthetic")


 PaySim Homophily 
  Edge Homophily: 0.9748
  High clustering (Nodes connect to same class). GNNs usually excel here.

 Elliptic Homophily 
  Edge Homophily: 0.9537
  High clustering (Nodes connect to same class). GNNs usually excel here.

 Synthetic Homophily 
  Edge Homophily: 0.3131
  Mixed connectivity.


## 3. Splitting Strategies

In [8]:
def create_temporal_split(data):
    # Ensure timesteps exist
    if not hasattr(data, 'timesteps'):
        print("Error: Data object missing 'timesteps' attribute")
        return data

    # 1. Create Masks based on Time
    train_time_mask = data.timesteps <= 34
    test_time_mask = data.timesteps > 34
    
    # 2. Filter out Unknown Labels (-1)
    known_mask = data.y != -1
    
    # Combine logic
    data.train_mask = train_time_mask & known_mask
    data.test_mask = test_time_mask & known_mask
    
    val_time_mask = (data.timesteps >= 30) & (data.timesteps <= 34)
    data.val_mask = val_time_mask & known_mask
    
    # Adjust train mask to exclude validation for strictness
    data.train_mask = data.train_mask & (~data.val_mask)

    print("\n Elliptic Temporal Split ")
    print(f"  Train Nodes: {data.train_mask.sum():,}")
    print(f"  Val Nodes:   {data.val_mask.sum():,}")
    print(f"  Test Nodes:  {data.test_mask.sum():,}")
    
    return data

if elliptic_data:
    elliptic_data = create_temporal_split(elliptic_data)


 Elliptic Temporal Split 
  Train Nodes: 26,381
  Val Nodes:   3,513
  Test Nodes:  16,670


In [10]:
def create_stratified_split(data, name):
    y_np = data.y.numpy()
    indices = np.arange(data.num_nodes)
    
    # 1. First Split: Train+Val (80%) vs Test (20%)
    splitter_test = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_val_idx, test_idx = next(splitter_test.split(indices, y_np))
    
    # 2. Second Split: Train (85% of 80% ≈ 68%) vs Val (15% of 80% ≈ 12%)
    y_train_val = y_np[train_val_idx]
    splitter_val = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    train_idx_sub, val_idx_sub = next(splitter_val.split(train_val_idx, y_train_val))
    
    # Map subset indices back to original indices
    train_idx = train_val_idx[train_idx_sub]
    val_idx = train_val_idx[val_idx_sub]
    
    # 3. Create Boolean Masks
    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    
    data.train_mask = train_mask
    data.val_mask = val_mask
    data.test_mask = test_mask
    
    print(f"\n{name} Stratified Split")
    print(f"  Train: {train_mask.sum():,} ({train_mask.sum()/data.num_nodes:.1%})")
    print(f"  Val:   {val_mask.sum():,}   ({val_mask.sum()/data.num_nodes:.1%})")
    print(f"  Test:  {test_mask.sum():,}  ({test_mask.sum()/data.num_nodes:.1%})")
    
    # Verify Fraud Distribution
    train_fraud = data.y[train_mask].sum().item()
    train_total = train_mask.sum().item()
    print(f"  Train Fraud Rate: {train_fraud/train_total:.2%}")
    
    return data

if paysim_data:
    paysim_data = create_stratified_split(paysim_data, "PaySim")

if synthetic_data:
    synthetic_data = create_stratified_split(synthetic_data, "Synthetic")


PaySim Stratified Split
  Train: 402,755 (68.0%)
  Val:   71,075   (12.0%)
  Test:  118,458  (20.0%)
  Train Fraud Rate: 1.39%

Synthetic Stratified Split
  Train: 3,400 (68.0%)
  Val:   600   (12.0%)
  Test:  1,000  (20.0%)
  Train Fraud Rate: 9.85%


## 4. Final Export

In [11]:
# Define save paths
save_paths = {
    "elliptic_final.pt": elliptic_data,
    "paysim_final.pt": paysim_data,
    "synthetic_final.pt": synthetic_data
}

for filename, data_obj in save_paths.items():
    if data_obj is not None:
        save_path = DATA_PROCESSED / filename
        
        # Security check: Ensure masks exist
        if hasattr(data_obj, 'train_mask'):
            torch.save(data_obj, save_path)
            print(f" Saved {filename}")
            
            # Print structure for verification
            print(f"   Structure: {data_obj}")
        else:
            print(f"  Skipping {filename}: Missing train_mask.")

print("\n Datasets are ready for GNN training.")

 Saved elliptic_final.pt
   Structure: Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timesteps=[203769], num_nodes=203769, train_mask=[203769], test_mask=[203769], val_mask=[203769])
 Saved paysim_final.pt
   Structure: Data(x=[592288, 8], edge_index=[2, 325933], y=[592288], num_nodes=592288, train_mask=[592288], val_mask=[592288], test_mask=[592288])
 Saved synthetic_final.pt
   Structure: Data(x=[5000, 12], edge_index=[2, 40900], edge_attr=[40900, 5], y=[5000], train_mask=[5000], val_mask=[5000], test_mask=[5000])

 Datasets are ready for GNN training.
